In [0]:


from pyspark.sql import functions as F
from pyspark.sql.functions import current_timestamp, lit

print("Bronze Notebook Started")


storage_account = "retailadls67"
access_key = "

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",access_key
)


PRODUCTS_PATH = f"abfss://landing@{storage_account}.dfs.core.windows.net/csv/products.csv"
ORDERS_PATH = f"abfss://landing@{storage_account}.dfs.core.windows.net/csv/orders.csv"
CUSTOMERS_PATH = f"abfss://landing@{storage_account}.dfs.core.windows.net/sql/customers.csv"
EXCHANGE_PATH = f"abfss://landing@{storage_account}.dfs.core.windows.net/api/exchange_rates.json"


BRONZE_PRODUCTS = f"abfss://bronze@{storage_account}.dfs.core.windows.net/products"

BRONZE_ORDERS = f"abfss://bronze@{storage_account}.dfs.core.windows.net/orders"

BRONZE_CUSTOMERS = f"abfss://bronze@{storage_account}.dfs.core.windows.net/customers"

BRONZE_EXCHANGE = f"abfss://bronze@{storage_account}.dfs.core.windows.net/exchange_rates"

print("Storage Connected")

In [0]:
products_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(PRODUCTS_PATH)
)

products_df = (
    products_df
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_file", lit("products.csv"))
)

products_df.show(5)

In [0]:
orders_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(ORDERS_PATH)
)

orders_df = (
    orders_df
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_file", lit("orders.csv"))
)

orders_df.show(5)

In [0]:
customers_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(CUSTOMERS_PATH)
)

customers_df = (
    customers_df
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_file", lit("customers.csv"))
)

customers_df.show(5)

In [0]:
exchange_df = (
    spark.read
    .option("multiline", True)
    .json(EXCHANGE_PATH)
)

exchange_df = (
    exchange_df
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_file", lit("exchange_rates.json"))
)

exchange_df.show(5)

In [0]:
products_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(BRONZE_PRODUCTS)

orders_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(BRONZE_ORDERS)

customers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(BRONZE_CUSTOMERS)

exchange_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(BRONZE_EXCHANGE)

In [0]:
orders_df.display()

In [0]:


print("Products :", products_df.count())
print("Orders :", orders_df.count())
print("Customers :", customers_df.count())
print("Exchange :", exchange_df.count())

